In [1]:
import numpy as np
import peaq_numpy
import csv
import scipy.io
import os
import peaq_numpy
from tqdm import tqdm
import librosa

In [2]:
with open("ds_test_2fmodel_pqevalaudio/ReferenceNumbers.csv", newline="") as csvfile:
	reader = csv.reader(csvfile, delimiter=";")
	# for row in reader:
		# print(row)



In [3]:
def get_dict_from_csv(path="ds_test_2fmodel_pqevalaudio/ReferenceNumbers.csv"):
	out_list=[]
	with open(path, newline="") as csvfile:
		reader = list(csv.reader(csvfile, delimiter=";"))
		firstline = reader[0]
		for row in reader[1:]:
			obj_dict = {
				firstline[0]: row[0][2:],
				firstline[1]: row[1][2:],
				firstline[2]: float(row[2])
			}
			out_list.append(obj_dict)
	return out_list
			# print(row)


In [4]:
all_files = get_dict_from_csv()

In [5]:
audio_path = "ds_test_2fmodel_pqevalaudio/SASSEC"
peaq = peaq_numpy.PEAQ(verbose=False, Amax=1)

M=0
for file in tqdm(all_files):
	if "anchor" in file["Test_Signal"]:
		test_sig=file["Test_Signal"]
		test_sig = test_sig[15:]
		test_sig = "Signals/anker_mix/"+test_sig
	else:
		test_sig=file["Test_Signal"]
	rate, x_R=librosa.load(os.path.join(audio_path, file["Reference_Signal"]), sr=None)
	rate, x_T=librosa.load(os.path.join(audio_path, test_sig), sr=None)
	
	M = max(np.amax(x_R), M)
	M = max(np.amax(x_T), M)
print(M)

  0%|          | 0/182 [00:00<?, ?it/s]

100%|██████████| 182/182 [00:06<00:00, 27.23it/s]

48000


In [6]:
audio_path = "ds_test_2fmodel_pqevalaudio/SASSEC"
peaq = peaq_numpy.PEAQ(verbose=False, Amax=1)

list_computed=[]
list_expected = []
for file in tqdm(all_files):
	if "anchor" in file["Test_Signal"]:
		test_sig=file["Test_Signal"]
		test_sig = test_sig[15:]
		test_sig = "Signals/anker_mix/"+test_sig
	else:
		test_sig=file["Test_Signal"]
	# rate, x_R=scipy.io.wavfile.read(os.path.join(audio_path, file["Reference_Signal"]))
	# rate, x_T=scipy.io.wavfile.read(os.path.join(audio_path, test_sig))
	x_R, rate=librosa.load(os.path.join(audio_path, file["Reference_Signal"]), sr=None, mono=False)
	x_T, rate=librosa.load(os.path.join(audio_path, test_sig), sr=None, mono=False)
	# print(x_R.shape)
	MMS_computed = peaq.compute_2fmodel_from_waveform(x_T, x_R)
	MMS_expected = file["Estimated_Mean_MUSHRA_Score"]
	list_computed.append(MMS_computed)
	list_expected.append(MMS_expected)

list_computed = np.array(list_computed)
list_expected = np.array(list_expected)

100%|██████████| 182/182 [02:50<00:00,  1.07it/s]


In [13]:
# for i in range(len(list_computed)):
    # print(list_computed[i], list_expected[i])
print(np.sqrt(np.mean(np.square(list_computed-list_expected))))
print(np.amax(np.abs(list_computed-list_expected)))

0.08615472842001999
0.721777322994825
